# What moves an unstable mode?

A spectrum answers *where* the modes sit.
**Eigenvalue sensitivities** answer *what moves them*: which duct length, flame lag, or feed condition pulls each mode toward or away from the stability boundary — without re-solving the mean flow or re-searching the spectrum for every candidate.

This notebook takes the classic self-excited [Rijke tube](rijke_tube.ipynb) and uses `modes.sensitivities()` to rank every setup parameter by how strongly it changes each mode's growth rate.
The dedicated bar chart (`sens.plot()`) makes the ranking readable at a glance, and a short `eigenvalue_trajectory` confirms that the local slope predicts a finite design change.


In [ ]:
import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.elements import catalog as cat
from nefes.elements import n_tau_flame
from nefes.perturbation import PerturbationBC
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

## 1. The tube

Cold air enters a rigid inlet ($\dot{m}'=0$), crosses a compact heat-release flame, and leaves through an open end.
An $n$-$\tau$ flame transfer function closes the thermoacoustic feedback.
The geometry and operating point match the self-excited Rijke example, so the unstable fundamental is already familiar.


In [ ]:
# Perfect gas
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)

# Geometry and mean flow
AREA = 0.01
L_COLD, L_HOT = 0.6, 0.4
MDOT = 0.006
T_IN, P_OUT = 300.0, 1.0e5

# Compact flame: prescribed total-temperature rise
DT_RISE = 400.0
QDOT = MDOT * CP * DT_RISE

# Flame transfer function and search box
N, TAU = 0.8, 4.0e-3
FREQ_BAND, GROWTH_BAND = (40.0, 320.0), (-200.0, 200.0)


# Build once: inlet -> cold duct -> compact flame -> hot duct -> open end.
net = nefes.Network(nefes.perfect_gas(R, GAMMA))
inlet = net.add(cat.mass_flow_inlet(MDOT, T_IN, name="inlet", perturbation_bc=PerturbationBC.hard_wall()))
cold = net.add(cat.duct(L_COLD, name="cold"))
flame = net.add(cat.heat_release_flame(QDOT, name="flame"))
hot = net.add(cat.duct(L_HOT, name="hot"))
outlet = net.add(cat.pressure_outlet(P_OUT, name="outlet", perturbation_bc=PerturbationBC.mean_flow_open_end()))

net.connect(inlet, cold, area=AREA)
ref = net.connect(cold, flame, area=AREA)  # velocity reference for the FTF
net.connect(flame, hot, area=AREA)
net.connect(hot, outlet, area=AREA)

net.set_dynamic_source(flame, n_tau_flame(N, TAU, ref_edge=ref))

sol = net.solve()
assert sol.converged, (sol.residual_norm, sol.print_residuals())
sol.verify()
sol.print_states()
sol.plot(color_by="T", title="Rijke tube: mean temperature").show()

## 2. The spectrum to differentiate

The certified eigenmode search finds the free oscillations of the active tube.
One mode is unstable; that is the design target the sensitivities will rank levers against.


In [ ]:
modes = sol.eigenmodes(freq_band=FREQ_BAND, growth_band=GROWTH_BAND, isentropic=True)

assert modes.certified, "search region needs a completeness certificate for a clean ranking"
for m in modes.summary():
    flag = "  <-- UNSTABLE" if m["unstable"] else ""
    print(f"f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s{flag}")

i_dom = int(np.argmax(modes.growth_rates))
print(f"\ndominant mode index: {i_dom}")

modes.plot_spectrum(title=f"Rijke spectrum  (n={N}, τ={TAU*1e3:.0f} ms)").show()

## 3. Rank every setup parameter

`modes.sensitivities()` differentiates every found mode against the network's parameter inventory in one pass.
Each parameter costs one operator re-assembly (plus a mean-flow chain term when it reshapes the steady state) — no re-solve, no re-search.

The result is a mode-by-parameter table.
A positive growth-rate entry means *increasing* that parameter pushes the mode toward instability.
The printed ranking and `sens.plot()` both scale each column to a +1% change of the parameter (or one probe step when the base value is zero), so levers with different units become comparable.


In [ ]:
sens = modes.sensitivities()

sens  # ranked table: growth-rate change per +1% (positive = destabilizing)

In [ ]:
print("most influential overall:", sens.top(5))
print(f"most influential for the unstable mode (#{i_dom}):", sens.top(5, mode=i_dom))

# The dedicated bar chart: red bars destabilize, blue bars stabilize.
sens.plot(
    modes=[i_dom] + [i for i in range(modes.n_modes) if i != i_dom],
    top=8,
    title="What moves each mode's growth rate?",
).show()

### Reading the ranking

For the unstable fundamental, **lengthening the cold duct** is the strongest stabilizing lever: a +1% increase in `cold.length` cuts the growth rate by several 1/s.
Raising the flame gain $n$ or the time lag $\tau$ does the opposite — both push the mode further into the unstable half-plane.

The second (stable) mode tells a different story: it is almost insensitive to `cold.length`, and instead reacts strongly to `hot.length` and to $\tau$.
That contrast is the practical payoff of the table — the same physical knob can stabilize one mode and leave another untouched, or even hurt it.


## 4. Isolate the flame knobs

Parameters tagged `layer="perturbation"` never touch the mean flow (FTF gain and lag, acoustic volumes, boundary closures).
Narrowing to that layer skips the mean-flow chain term automatically and answers a sharper question: *holding the operating point fixed, what does retuning the flame response do?*


In [ ]:
sens_ftf = modes.sensitivities(layer="perturbation")

print(sens_ftf)
sens_ftf.plot(modes=i_dom, top=6, title="Perturbation-layer levers on the unstable mode").show()

## 5. Confirm the local slope with a trajectory

Sensitivities are first-order slopes.
A finite design change should be checked by continuation along the chosen address — here the top stabilizer, `cold.length`.

The dashed line is the local linear prediction from `sens.dgrowth_dp`; the markers are the continued eigenvalues.
Near the base length they agree, and lengthening the cold duct drives the growth rate toward the stability boundary exactly as the ranking promised.


In [ ]:
# Local slope of the dominant mode vs cold-duct length
addr = "cold.length"
k = sens.addresses.index(addr)
slope_g = float(sens.dgrowth_dp[i_dom, k])
slope_f = float(sens.dfreq_dp[i_dom, k])
f0 = float(modes.freqs[i_dom])
g0 = float(modes.growth_rates[i_dom])

L_sweep = np.linspace(0.55, 0.70, 13)
traj = sol.network.eigenvalue_trajectory(
    addr,
    L_sweep,
    freq_band=FREQ_BAND,
    growth_band=GROWTH_BAND,
    isentropic=True,
)

# Match the continued branch to the dominant base mode
branch = min(traj.branches, key=lambda b: abs(b.freqs[np.argmin(np.abs(L_sweep - L_COLD))] - f0))
L_line = np.linspace(L_sweep.min(), L_sweep.max(), 80)

fig = go.Figure()
fig.add_scatter(
    x=L_sweep,
    y=branch.growth,
    mode="markers+lines",
    name="eigenvalue trajectory",
    marker=dict(size=9, color=COLORWAY[4]),
    line=dict(color=COLORWAY[4], width=1.5),
)
fig.add_scatter(
    x=L_line,
    y=g0 + slope_g * (L_line - L_COLD),
    mode="lines",
    name="sensitivity tangent",
    line=dict(color=COLORWAY[0], width=2, dash="dash"),
)
fig.add_hline(y=0.0, line=dict(color="#9aa5b1", width=1.2, dash="dot"))
fig.add_vline(x=L_COLD, line=dict(color="#9aa5b1", width=1.2, dash="dot"))
fig.update_layout(
    title="Confirming the top stabilizer: cold-duct length",
    xaxis_title="cold.length [m]",
    yaxis_title="growth rate [1/s]",
    width=900,
    height=420,
)
fig.show()

# Spot-check: +5% cold length, predicted vs re-solved
dL = 0.05 * L_COLD
g_pred = g0 + slope_g * dL
f_pred = f0 + slope_f * dL
sol_pert = net.with_params({"cold.length": L_COLD + dL}).solve(x0=sol.x)
modes_pert = sol_pert.eigenmodes(freq_band=FREQ_BAND, growth_band=GROWTH_BAND, isentropic=True)
j = int(np.argmin(np.abs(modes_pert.freqs - f0)))
print(f"+5% cold.length on mode #{i_dom}:")
print(f"  sensitivity predicts  f = {f_pred:7.2f} Hz,  growth = {g_pred:+8.2f} 1/s")
print(f"  re-solved eigenmode   f = {modes_pert.freqs[j]:7.2f} Hz,  growth = {modes_pert.growth_rates[j]:+8.2f} 1/s")

## Summary

* `modes.sensitivities()` turns a certified spectrum into a ranked table of design levers in one pass — far cheaper than a re-solve / re-search per parameter.
* `sens.plot()` draws that ranking as a bar chart: red bars destabilize, blue bars stabilize, hover text carries the frequency shift.
* On this Rijke tube the unstable fundamental is controlled primarily by **`cold.length`** (stabilizing) and the flame knobs **`n` / `τ`** (destabilizing); the second mode answers to a different set of knobs.
* The derivatives are local slopes: confirm a finite change with `eigenvalue_trajectory` (or a single re-solve) along the address you intend to move.
